# Sample-group evaluation note

This notebook is retained to reproduce the second sample-group evaluation workflow. It performs a `GroupShuffleSplit` from the combined augmented dataset within the notebook and should not be interpreted as the canonical preprocessing or training implementation.


### Prediction with Group-based Splitting

This notebook demonstrates the process of predicting input values using pre-trained models from `forward_model_new.ipynb` and `diffusion_model_new.ipynb`. The goal is to refine predictions to achieve desired target values.

#### Key Features:
- **Group-based Data Splitting**: Uses `GroupShuffleSplit` to prevent data leakage (Ep1-Epp3 columns repeat every 130 rows)
- **Augmented Data**: Uses `combined_augmented_all_data.xlsx` with new scalers (`scaler_x_new.pkl`, `scaler_y_new.pkl`)
- **Updated Models**: Uses `EnhancedDiffusionModel` and `ImprovedRegressionModel` with latest weights

#### Process:
1. **Model Loading**: Load pre-trained forward and diffusion models
2. **Data Preparation**: Load augmented data with group-based splitting
3. **Sampling**: Use diffusion model to sample candidates
4. **Visualization**: Plot results comparing before and after refinement

#### Compatibility:
- Matches `RNN_augmentation_new.ipynb` (data & scalers)
- Matches `diffusion_model_new.ipynb` (diffusion model)
- Matches `forward model_new.ipynb` (forward model)


#### Preparation and Environment Settings

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib, pandas as pd

In [ ]:
import matplotlib as mpl
mpl.rcParams['xtick.major.size'] = 8
mpl.rcParams['ytick.major.size'] = 8
mpl.rcParams['xtick.major.width'] = 3
mpl.rcParams['ytick.major.width'] = 3

In [ ]:
# Use serif + bold global settings before drawing
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

#### Load Models 

In [ ]:
# Load Models
base_dir = Path("")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Forward Model
class ImprovedRegressionModel(nn.Module):
    def __init__(self,
                 moduli_dim: int = 6,
                 dist_dim: int = 2,
                 hidden_dim: int = 256,
                 trunk_layers: int = 2,
                 dropout: float = 0.1):
        super().__init__()
        assert dist_dim == 2, "This model expects dist_dim=2"
        self.moduli_dim = moduli_dim
        self.dist_dim   = dist_dim

        trunk = []
        in_dim = moduli_dim
        for _ in range(trunk_layers):
            trunk.append(nn.Linear(in_dim, hidden_dim))
            trunk.append(nn.LayerNorm(hidden_dim))
            trunk.append(nn.GELU())
            trunk.append(nn.Dropout(dropout))
            in_dim = hidden_dim
        self.trunk = nn.Sequential(*trunk)

        self.dist_embed = nn.Sequential(
            nn.Linear(dist_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        fusion_dim = hidden_dim + (hidden_dim // 4)

        self.head0 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

        self.head1 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1)
        )

        self.log_sigma0 = nn.Parameter(torch.zeros(()))
        self.log_sigma1 = nn.Parameter(torch.zeros(()))

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_mod  = x[:, : self.moduli_dim]
        x_dist = x[:, self.moduli_dim:]

        z_mod = self.trunk(x_mod)
        z_dist = self.dist_embed(x_dist)

        z = torch.cat([z_mod, z_dist], dim=1)

        y0 = self.head0(z)
        y1 = self.head1(z)

        return torch.cat([y0, y1], dim=-1)

forward_model = ImprovedRegressionModel(moduli_dim=6, dist_dim=2, hidden_dim=256, trunk_layers=2, dropout=0.1).to(device)
forward_model.load_state_dict(torch.load(base_dir/"artifacts/soft_material/forward_model_weights_new.pt", map_location=device))
forward_model.eval()

In [ ]:
# Load Diffusion Model
class DiffusionSchedule(nn.Module):
    def __init__(self, T: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02, eps: float = 1e-8):
        super().__init__()
        self.T = T
        self.eps = eps

        beta = torch.linspace(beta_start, beta_end, T)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]], dim=0)

        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None) -> torch.Tensor:
        if noise is None:
            noise = torch.randn_like(x0[:, 6:])

        ab = self.alpha_bar.to(x0.device)[t].clamp(self.eps, 1.0 - self.eps)
        sqrt_ab = ab.sqrt().view(-1, 1)
        sqrt_omb = (1.0 - ab).sqrt().view(-1, 1)

        x_mod = x0[:, :6]
        x_dist = x0[:, 6:]
        x_dist_t = sqrt_ab * x_dist + sqrt_omb * noise

        return torch.cat([x_mod, x_dist_t], dim=1)

class TimeEmbedding(nn.Module):
    """Sinusoidal time embedding"""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        half = dim // 2
        factor = math.log(10000.0) / (half - 1)
        self.register_buffer('factor', torch.tensor(factor))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        emb_scale = torch.exp(-self.factor * torch.arange(half, device=t.device))
        emb = t[:, None].float() * emb_scale[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=1)

class EnhancedDiffusionModel(nn.Module):
    """Enhanced diffusion model: deeper network with better regularization"""
    def __init__(self, x_dim=8, y_dim=2, time_dim=32, hidden_dim=256):
        super().__init__()
        self.x_dim = x_dim

        # === Moduli branch (Ep1-Epp3): enhanced capacity ===
        self.moduli_embed = nn.Sequential(
            nn.Linear(6, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

        # === Distribution branch (t1, t2): deep MLP ===
        self.dist_mlp = nn.Sequential(
            nn.Linear(2, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

        # === Y conditioning branch (EpH, EppH) ===
        self.y_mlp = nn.Sequential(
            nn.Linear(y_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128)
        )

        # === Time embedding branch ===
        self.time_embed = TimeEmbedding(time_dim)
        self.time_fc = nn.Sequential(
            nn.Linear(time_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, 128)
        )

        # === Fusion layer: deep network ===
        fusion_dim = 128 + 128 + 128 + 128  # moduli + dist + y + time
        self.fusion_fc = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)  # Output noise prediction (t1, t2)
        )

    def forward(self, x, t, y):
        """Forward propagation
        x: [B, 8] - [Ep1, Ep2, Ep3, Epp1, Epp2, Epp3, t1_noisy, t2_noisy]
        t: [B] - timestep
        y: [B, 2] - condition [EpH, EppH]
        """
        x_mod = x[:, :6]   # Ep1-Epp3
        x_dist = x[:, 6:]  # t1, t2 (noisy)

        # Feature extraction from each branch
        mod_feat = self.moduli_embed(x_mod)
        dist_feat = self.dist_mlp(x_dist)
        y_feat = self.y_mlp(y)
        t_emb = self.time_embed(t)
        t_feat = self.time_fc(t_emb)

        # Fuse all features
        fused = torch.cat([mod_feat, dist_feat, y_feat, t_feat], dim=1)
        eps_pred = self.fusion_fc(fused)

        return eps_pred

diffusion_model = EnhancedDiffusionModel(x_dim=8, y_dim=2, time_dim=32, hidden_dim=384).to(device)
state_dict = torch.load(base_dir/"artifacts/soft_material/diffusion_model_new_best.pt", map_location=device)
diffusion_model.load_state_dict(state_dict)
diffusion_model.eval()

#### Data Preparation (based on TrainingData3.xlsx)

In [ ]:
# Data Preparation
import joblib, pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load new scalers (based on augmented data)
scaler_x = joblib.load("artifacts/soft_material/scaler_x_new.pkl")
scaler_y = joblib.load("artifacts/soft_material/scaler_y_new.pkl")

# Load augmented data
df = pd.read_excel("data/combined_augmented_all_data.xlsx", header=0)
X_raw = df.iloc[:, :8].values
Y_raw = df.iloc[:, 8:].values

# Define moduli columns for grouping (Ep1-Epp3 repeat every 130 rows)
MOD = ['Ep1', 'Ep2', 'Ep3', 'Epp1', 'Epp2', 'Epp3']
groups = pd.util.hash_pandas_object(df[MOD].round(6), index=False).astype(np.int64).to_numpy()

# Use GroupShuffleSplit to ensure samples from the same group stay together
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X_raw, Y_raw, groups=groups))

X_tr = X_raw[train_idx]
Y_tr = Y_raw[train_idx]
X_val = X_raw[val_idx]
Y_val = Y_raw[val_idx]

print(f"Total samples: {len(X_raw)}, Unique groups: {len(np.unique(groups))}")
print(f"Train set: {len(X_tr)} samples, {len(np.unique(groups[train_idx]))} groups")
print(f"Validation set: {len(X_val)} samples, {len(np.unique(groups[val_idx]))} groups")

# Check for data leakage
train_groups = groups[train_idx]
val_groups = groups[val_idx]
overlap = set(np.unique(train_groups)) & set(np.unique(val_groups))
if overlap:
    print(f"⚠️ Warning: Found {len(overlap)} overlapping groups!")
else:
    print("✓ No data leakage: Train and validation groups are completely separate")

In [ ]:
# Diffusion Schedule
schedule = DiffusionSchedule(T=1000).to(device)

# Sampling Function
@torch.no_grad()
def sample_with_fixed_moduli(diffusion_model, schedule, y_cond, moduli_fixed):
    x_dist = torch.randn(1, 2, device=device)
    x = torch.cat([moduli_fixed, x_dist], dim=1)

    for t in reversed(range(schedule.T)):
        t_tensor = torch.full((1,), t, device=device, dtype=torch.long)
        eps_dist = diffusion_model(x, t_tensor, y_cond)

        beta_t = schedule.beta[t]
        alpha_t = schedule.alpha[t]
        alpha_bar_t = schedule.alpha_bar[t]
        denom_sqrt = torch.sqrt((1 - alpha_bar_t).clamp(min=schedule.eps))

        x_mod_part = moduli_fixed
        x_dist_part = x[:, 6:]

        mu_dist = (1 / torch.sqrt(alpha_t.clamp(min=schedule.eps))) * (
            x_dist_part - (beta_t / denom_sqrt) * eps_dist
        )

        if t > 0:
            ab_prev = schedule.alpha_bar_prev[t]
            sigma_t = torch.sqrt(beta_t * (1 - ab_prev) / (1 - alpha_bar_t))
            z = torch.randn_like(mu_dist)
            x_dist_prev = mu_dist + sigma_t * z
        else:
            x_dist_prev = mu_dist

        x = torch.cat([x_mod_part, x_dist_prev], dim=1)

    return x


### Prediction

#### Sample 1 val_sample_idx = 1000 

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 1000  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(2, 4)
plt.ylim(1, 3)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 2 val_sample_idx = 1000 [5.0,1.2]

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 1000  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y
raw_target = np.array([[5.0,1.2]])
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(4, 6)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 3 val_sample_idx = 1030

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 1030  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(2, 4)
plt.ylim(1, 3)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 4 val_sample_idx = 1030 [4, 1.6]

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 1030  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y
raw_target = np.array([[4, 1.6]])
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(3, 5)
plt.ylim(1, 3)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 5 val_sample_idx = 1896

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 1896  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(0.5, 2.5)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 6  val_sample_idx = 1896  [1.65,1.0]

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 1896  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y
raw_target = np.array([[1.65, 1.0]])
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(0.5, 2.5)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 7 val_sample_idx = 98

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 98  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(0, 2)
plt.ylim(-0.5, 1.5)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 8 val_sample_idx = 198

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 198  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(2.5, 4.5)
plt.ylim(0.5, 2.5)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 9 val_sample_idx = 192

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 192  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(1, 3)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 10 val_sample_idx = 408

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 408  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(1.5, 3.5)
plt.ylim(1, 3)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 11 val_sample_idx = 2082

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 2082  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(3.5, 5.5)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 12 val_sample_idx = 3464

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 3464  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(4, 6)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 13 val_sample_idx = 4139

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 4139 # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(2, 4)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 14 val_sample_idx = 4188

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 4188  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(3.5, 5.5)
plt.ylim(0.5, 2.5)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 15 val_sample_idx = 4101

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 4101  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(2.5, 4.5)
plt.ylim(0.5, 2.5)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 16 val_sample_idx = 2122

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 2122  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(4.5, 6.5)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 17 val_sample_idx = 744

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 744  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(4.5, 6.5)
plt.ylim(-0.5, 1.5)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### Sample 18 val_sample_idx = 519

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 519  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y (take the last 2 columns from the same row in the original dataset)
raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

In [ ]:
# Sampling and Prediction
num_samples = 200
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier

def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

In [ ]:
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)

best_idx = np.argmin(errors_orig)

# Reuse the refined_* names so later code doesn't need changes
refined_x_orig = cands_x_orig[best_idx:best_idx+1]   # shape (1, 8) in original scale
refined_y_orig = preds_orig[best_idx:best_idx+1]     # shape (1, 2) in original scale
refined_mse    = float(errors_orig[best_idx])

# (Optional) keep these for reference if you want to compare
orig_x_orig = refined_x_orig.copy()
orig_y_orig = refined_y_orig.copy()

print("Chosen best candidate (no refinement):")
print(f"  x = {np.round(refined_x_orig.flatten(), 5)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 5)}")
print(f"  MSE = {refined_mse:.6f}")

In [ ]:
# Plot Candidates
errors = np.mean((preds_orig - raw_target)**2, axis=1)
top_idx = np.argsort(errors)[:50]

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=150, edgecolors='b', label='Best candidate')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(0, 2)
plt.ylim(0, 2)
plt.legend(prop={'size':24, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for All Candidates
plt.figure(figsize=(6, 6))
# Top 100 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=200)
# Best candidate
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='blue', s=200, edgecolors='k')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()